In [ ]:
import torch
import os, math
import numpy as np
from pathlib import Path
from scipy.spatial.transform import Rotation
import matplotlib.pyplot as plt
import collections

In [ ]:
def quaternion_matrix(quaternion):
    """Return homogeneous rotation matrix from quaternion.

    >>> R = quaternion_matrix([0.06146124, 0, 0, 0.99810947])
    >>> numpy.allclose(R, rotation_matrix(0.123, (1, 0, 0)))
    True

    """
    q = np.array(quaternion[:4], dtype=np.float64, copy=True)
    nq = np.dot(q, q)
    if nq < 1e-9:
        return np.identity(4)
    q *= math.sqrt(2.0 / nq)
    q = np.outer(q, q)
    return np.array((
        (1.0-q[1, 1]-q[2, 2],     q[0, 1]-q[2, 3],     q[0, 2]+q[1, 3], 0.0),
        (    q[0, 1]+q[2, 3], 1.0-q[0, 0]-q[2, 2],     q[1, 2]-q[0, 3], 0.0),
        (    q[0, 2]-q[1, 3],     q[1, 2]+q[0, 3], 1.0-q[0, 0]-q[1, 1], 0.0),
        (                0.0,                 0.0,                 0.0, 1.0)
        ), dtype=np.float64)

def quaternion_from_matrix(matrix):
    """Return quaternion from rotation matrix.

    >>> R = rotation_matrix(0.123, (1, 2, 3))
    >>> q = quaternion_from_matrix(R)
    >>> numpy.allclose(q, [0.0164262, 0.0328524, 0.0492786, 0.9981095])
    True

    """
    q = np.empty((4, ), dtype=np.float64)
    M = np.array(matrix, dtype=np.float64, copy=False)[:4, :4]
    t = np.trace(M)
    if t > M[3, 3]:
        q[3] = t
        q[2] = M[1, 0] - M[0, 1]
        q[1] = M[0, 2] - M[2, 0]
        q[0] = M[2, 1] - M[1, 2]
    else:
        i, j, k = 0, 1, 2
        if M[1, 1] > M[0, 0]:
            i, j, k = 1, 2, 0
        if M[2, 2] > M[i, i]:
            i, j, k = 2, 0, 1
        t = M[i, i] - (M[j, j] + M[k, k]) + M[3, 3]
        q[i] = t
        q[j] = M[i, j] + M[j, i]
        q[k] = M[k, i] + M[i, k]
        q[3] = M[k, j] - M[j, k]
    q *= 0.5 / math.sqrt(t * M[3, 3])
    return q

def colmap_to_robot(t_colmap, q_colmap):
    q = np.roll(q_colmap, -1)
    R = quaternion_matrix(q).T
    t = R[:3, :3]@(-t_colmap)
    return t, np.roll(quaternion_from_matrix(R), 1)
def robot_to_colmap(t_robot, q_robot):
    r = np.eye(4)
    r[:3, :3] = Rotation.from_quat(q_robot).as_matrix().T
    t_colmap = -r[:3, :3] @ t_robot
    q_colmap = np.roll(quaternion_from_matrix(r), 1)
    return t_colmap, q_colmap

##################################
# NERF PAPER (https://github.com/cmusatyalab/mega-nerf/blob/main/scripts/colmap_to_mega_nerf.py)
RDF_TO_DRB = torch.FloatTensor([[0, 1, 0],
                                [1, 0, 0],
                                [0, 0, -1]])
def qvec2rotmat(qvec):
    return np.array([
        [1 - 2 * qvec[2] ** 2 - 2 * qvec[3] ** 2,
         2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3],
         2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
        [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3],
         1 - 2 * qvec[1] ** 2 - 2 * qvec[3] ** 2,
         2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
        [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2],
         2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1],
         1 - 2 * qvec[1] ** 2 - 2 * qvec[2] ** 2]])

BaseImage = collections.namedtuple(
    "Image", ["id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids"])

class Image(BaseImage):
    def qvec2rotmat(self):
        return qvec2rotmat(self.qvec)

def read_images_text(path):
    """
    see: src/base/reconstruction.cc
        void Reconstruction::ReadImagesText(const std::string& path)
        void Reconstruction::WriteImagesText(const std::string& path)
    """
    images = {}
    with open(path, "r") as fid:
        while True:
            line = fid.readline()
            if not line:
                break
            line = line.strip()
            if len(line) > 0 and line[0] != "#":
                elems = line.split()
                image_id = int(elems[0])
                qvec = np.array(tuple(map(float, elems[1:5])))
                tvec = np.array(tuple(map(float, elems[5:8])))
                camera_id = int(elems[8])
                image_name = elems[9]
                elems = fid.readline().split()
                xys = np.column_stack([tuple(map(float, elems[0::3])),
                                       tuple(map(float, elems[1::3]))])
                point3D_ids = np.array(tuple(map(int, elems[2::3])))
                images[image_id] = Image(
                    id=image_id, qvec=qvec, tvec=tvec,
                    camera_id=camera_id, name=image_name,
                    xys=xys, point3D_ids=point3D_ids)
    return images
##################################


DRB_TO_RDF = torch.inverse(RDF_TO_DRB)


In [ ]:
images =  read_images_text("/media/fractal/Storage/ejaealb/abisko_saveup/copla-match/datasets/tum_testing/mapping/input_model/images.txt")
c2ws, outs, orig = {}, {}, {}
scale = 10
for image in images.values():
    w2c = torch.eye(4)
    w2c[:3, :3] = torch.FloatTensor(qvec2rotmat(image.qvec))
    w2c[:3, 3] = torch.FloatTensor(image.tvec)
    c2w = torch.inverse(w2c)

    c2w = torch.hstack((
        RDF_TO_DRB @ c2w[:3, :3] @ torch.inverse(RDF_TO_DRB),
        RDF_TO_DRB @ c2w[:3, 3:]
    ))

    c2ws[image.id] = c2w
    
    orig[image.id] = [image.qvec, image.tvec]

positions = torch.cat([c2w[:3, 3].unsqueeze(0) for c2w in c2ws.values()])
print('{} images'.format(positions.shape[0]))
max_values = positions.max(0)[0]
min_values = positions.min(0)[0]
origin = ((max_values + min_values) * 0.5)
dist = (positions - origin).norm(dim=-1)
diagonal = dist.max()
print(origin, diagonal, max_values, min_values)

for k in c2ws.keys():
    camera_in_drb = c2ws[k]
    camera_in_drb[:, 3] = (camera_in_drb[:, 3] - origin) / scale
    outs[k] = torch.cat([camera_in_drb[:, 1:2], -camera_in_drb[:, :1], camera_in_drb[:, 2:4]],-1)

In [ ]:
for k in c2ws.keys():    
    # NERF to COLMAP
    vv = torch.cat([-outs[k][:, 1:2], outs[k][:, :1], outs[k][:, 2:4]], -1)
    vv[:, 3] = vv[:, 3] *scale + origin
    a = torch.eye(4)
    a[:3] = torch.hstack((
        DRB_TO_RDF @ vv[:3, :3] @ torch.inverse(DRB_TO_RDF),
        DRB_TO_RDF @ vv[:3, 3:]
    ))
    a = torch.inverse(a).numpy()
    if not (np.allclose(a[:3, 3], orig[k][1]) and np.allclose(np.roll(Rotation.from_matrix(a[:3,:3]).as_quat(), 1), orig[k][0])):
        print("FAIL")
        print(orig[k][1], a[:3, 3])
        print(orig[k][0], np.roll(Rotation.from_matrix(a[:3,:3]).as_quat(), 1))

In [ ]:
base_path = Path("/home/ejaealb/work/hloc/datasets/sfmreg/Quad/ArtsQuad_dataset/")
info_path = Path("/home/ejaealb/work/hloc/datasets/sfmreg/Quad/quad-pixsfm/")
keys = []
image_dir =  Path(base_path, "images")
global_info = torch.load(info_path / "coordinates.pt")
factor = global_info["pose_scale_factor"]
origin = global_info["origin_drb"].numpy()

In [ ]:
images, poses, params = [], [], []
with open(info_path / "mappings.txt") as f:
    for line in f.readlines():
        x = line.strip().split(",")
        if os.path.exists(info_path / "train/metadata" / x[1]): info = torch.load(info_path / "train/metadata" / x[1])
        else: info = torch.load(info_path / "val/metadata" / x[1])

        out = info["c2w"]
        # NERF to COLMAP
        vv = torch.cat([-out[:, 1:2], out[:, :1], out[:, 2:4]], -1)
        vv[:, 3] = vv[:, 3] *factor + origin
        a = torch.eye(4)
        a[:3] = torch.hstack((
            DRB_TO_RDF @ vv[:3, :3] @ torch.inverse(DRB_TO_RDF),
            DRB_TO_RDF @ vv[:3, 3:]
        ))
        a = torch.inverse(a).numpy()
        t, q = a[:3, 3], np.roll(Rotation.from_matrix(a[:3,:3]).as_quat(), 1)
        poses.append(np.concatenate([q, t]))
        #map_[x[0]] = x[1]
        params.append(torch.cat([torch.Tensor([info["W"], info["H"]]), info["intrinsics"], info["distortion"]]).numpy())
        images.append(x[0])
poses = np.array(poses)

In [ ]:
print(info["H"], info["W"], info["intrinsics"], info["distortion"])

os.makedirs(base_path / "input_model", exist_ok=True)
with open(base_path / "input_model" / "images.txt", "w") as f:
    for i in range(len(images)):
        f.write(f"{i} " + " ".join(["%.20f"%x for x in poses[i]]) + f" {i} {images[i]}\n\n")
with open(base_path / "input_model" / "cameras.txt", "w") as f:
    for i in range(len(images)):
        f.write(f"{i} SIMPLE_RADIAL {int(params[i][0])} {int(params[i][1])} {round(params[i][2], 4)} {round(params[i][4], 4)} {round(params[i][5], 4)} {round(params[i][6], 4)}" + "\n")
with open(base_path / "input_model" / "points3D.txt", "w") as f:
    pass

In [ ]:
posos = []
for x in poses:
    t, q = colmap_to_robot(x[4:], x[:4])
    posos.append(np.concatenate([q, t]))
posos = np.array(posos)
f, ax = plt.subplots(ncols=2, figsize=(10, 5))
ax[0].scatter(posos[:, 4], posos[:, 5])
ax[1].scatter(posos[:, 4], posos[:, 6])
plt.show()

plt.plot(posos[:, :4])
plt.show()

In [ ]:
import pycolmap
from hloc.utils import viz_3d


reference_model_path = Path("/home/ejaealb/work/hloc/datasets/sfmreg/Quad/ArtsQuad_dataset/sfm/sift")
ref_reconstruction = pycolmap.Reconstruction(reference_model_path)
print(ref_reconstruction)
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig, ref_reconstruction, color="rgba(255,0,0,0.5)", name="mapping", points_rgb=True
)
fig.show()

reference_model_path = Path("/home/ejaealb/work/hloc/datasets/sfmreg/Quad/ArtsQuad_dataset/sfm_tr/sift")
ref_reconstruction = pycolmap.Reconstruction(reference_model_path)
print(ref_reconstruction)
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig, ref_reconstruction, color="rgba(255,0,0,0.5)", name="mapping", points_rgb=True
)
fig.show()